<a href="https://colab.research.google.com/github/nagaraj732007-del/fake-news-detection/blob/main/Household_Electricity_Prediction_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving household_power_consumption.csv to household_power_consumption (1).csv


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

print("All libraries imported successfully!")

All libraries imported successfully!


In [ ]:
filename = "household_power_consumption.csv"

filename = "household_power_consumption.csv"

df = pd.read_csv(
    filename,
    na_values=["?"],
    low_memory=False
)

# Remove an unnecessary index column if present
if "index" in df.columns:
    df = df.drop(columns=["index"])

print("Dataset loaded successfully!")

print("\nFirst five records:")
display(df.head())

print("\nDataset shape:")
print(df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isnull().sum())

Dataset loaded successfully!

First five records:


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,1/1/07,0:00:00,2.580,0.136,241.97,10.6,0.0,0.0,0.0
1,1/1/07,0:01:00,2.552,0.100,241.75,10.4,0.0,0.0,0.0
2,1/1/07,0:02:00,2.550,0.100,241.64,10.4,0.0,0.0,0.0
3,1/1/07,0:03:00,2.550,0.100,241.71,10.4,0.0,0.0,0.0
4,1/1/07,0:04:00,2.554,0.100,241.98,10.4,0.0,0.0,0.0



Dataset shape:
(260640, 9)

Column names:
['Date', 'Time', 'Global_active_power', 'Global_reactive_power', 'Voltage', 'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']

Missing values:
Date                        0
Time                        0
Global_active_power      3771
Global_reactive_power    3771
Voltage                  3771
Global_intensity         3771
Sub_metering_1           3771
Sub_metering_2           3771
Sub_metering_3           3771
dtype: int64


In [ ]:
# Combine Date and Time
df["Datetime"] = pd.to_datetime(
    df["Date"].astype(str) + " " + df["Time"].astype(str),
    dayfirst=True,
    errors="coerce"
)

# Sort records by time
df = df.dropna(subset=["Datetime"])
df = df.sort_values("Datetime")
df = df.drop_duplicates(subset=["Datetime"])
df = df.set_index("Datetime")

# Convert measurement columns to numeric
feature_columns = [
    "Global_active_power",
    "Global_reactive_power",
    "Voltage",
    "Global_intensity",
    "Sub_metering_1",
    "Sub_metering_2",
    "Sub_metering_3"
]

for col in feature_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Select an initial contiguous block of 10,000 minutes
df = df.iloc[:10000].copy()

# Fill missing measurements using time-based interpolation
df[feature_columns] = df[feature_columns].interpolate(
    method="time",
    limit_area="inside"
)

# Fill any remaining missing values
df[feature_columns] = df[feature_columns].ffill().bfill()

# Confirm there are no missing feature values
print("Missing values after cleaning:")
print(df[feature_columns].isnull().sum())

print("\nSelected records:", len(df))

print("\nFirst selected record:")
print(df.index[0])

print("\nLast selected record:")
print(df.index[-1])

# Check timestamp continuity
minute_differences = df.index.to_series().diff().dropna()

print(
    "\nNumber of gaps or irregular intervals:",
    (minute_differences != pd.Timedelta(minutes=1)).sum()
)

display(df.head())

/tmp/ipykernel_740/1121814883.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Datetime"] = pd.to_datetime(


Missing values after cleaning:
Global_active_power      0
Global_reactive_power    0
Voltage                  0
Global_intensity         0
Sub_metering_1           0
Sub_metering_2           0
Sub_metering_3           0
dtype: int64

Selected records: 10000

First selected record:
2007-01-01 00:00:00

Last selected record:
2007-01-07 22:39:00

Number of gaps or irregular intervals: 0


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
Datetime,,,,,,,,,
2007-01-01 00:00:00,1/1/07,0:00:00,2.580,0.136,241.97,10.6,0.0,0.0,0.0
2007-01-01 00:01:00,1/1/07,0:01:00,2.552,0.100,241.75,10.4,0.0,0.0,0.0
2007-01-01 00:02:00,1/1/07,0:02:00,2.550,0.100,241.64,10.4,0.0,0.0,0.0
2007-01-01 00:03:00,1/1/07,0:03:00,2.550,0.100,241.71,10.4,0.0,0.0,0.0
2007-01-01 00:04:00,1/1/07,0:04:00,2.554,0.100,241.98,10.4,0.0,0.0,0.0


In [ ]:
data = df[feature_columns].values

# Chronological split
n = len(data)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

# Fit the scaler only on the training data
scaler = MinMaxScaler()

scaler.fit(data[:train_end])

scaled_data = scaler.transform(data)

print("Normalization completed!")

print("Training records:", train_end)
print("Validation records:", val_end - train_end)
print("Testing records:", n - val_end)

Normalization completed!
Training records: 7000
Validation records: 1500
Testing records: 1500


In [ ]:
sequence_length = 60

# Global_active_power is the first feature
target_index = 0

def create_sequences(data, start, end):
    X, y = [], []

    for i in range(start, end):
        X.append(data[i - sequence_length:i])
        y.append(data[i, target_index])

    return np.array(X), np.array(y)


X_train, y_train = create_sequences(
    scaled_data,
    sequence_length,
    train_end
)

X_val, y_val = create_sequences(
    scaled_data,
    train_end,
    val_end
)

X_test, y_test = create_sequences(
    scaled_data,
    val_end,
    n
)

print("Training input:", X_train.shape)
print("Training target:", y_train.shape)

print("Validation input:", X_val.shape)
print("Testing input:", X_test.shape)

Training input: (6940, 60, 7)
Training target: (6940,)
Validation input: (1500, 60, 7)
Testing input: (1500, 60, 7)
